# 02 - Data Preprocessing

## Setups and Imports

In [ ]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / 'src').exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / 'src').exists():
    PROJECT_DIR = CURRENT_DIR.parent

SRC_DIR = PROJECT_DIR / 'src'

for dir in [PROJECT_DIR, SRC_DIR]:
    if str(dir) not in sys.path:
        sys.path.append(str(dir))

print(f'Project dir: {PROJECT_DIR}')
print(f'src dir: {SRC_DIR}')

In [ ]:
import json, random
import os, shutil
from IPython.display import HTML, Image, display

import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.utils.image_io import (
    convert_to_binary,
    save_grayscale_image
)
from src.utils.mask_utils import (
    mask_to_yolo
)

In [ ]:
DATA_INTERIM_DIR = PROJECT_DIR / 'data' / 'data_interim'
REPORTS_DIR = PROJECT_DIR / 'reports'

## Helper Functions

In [ ]:
def sample_video_and_mask_frames(
    video_path: Path,
    mask_path: Path,
    output_images_dir: Path,
    output_masks_dir: Path,
    sample_every_seconds: float = 2.0,
    image_ext: str = ".png",
    mask_ext: str = ".png",
    prefix: str = '',
):
    """
    Extract matching frames from an RGB video and its corresponding mask video.
    Saves frames using the same sample index so image/mask pairs stay aligned.
    """

    video_path = Path(video_path)
    mask_path = Path(mask_path)
    output_images_dir = Path(output_images_dir)
    output_masks_dir = Path(output_masks_dir)

    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    if not mask_path.exists():
        raise FileNotFoundError(f"Mask video not found: {mask_path}")

    output_images_dir.mkdir(parents=True, exist_ok=True)
    output_masks_dir.mkdir(parents=True, exist_ok=True)

    video_cap = cv2.VideoCapture(str(video_path))
    mask_cap = cv2.VideoCapture(str(mask_path))

    if not video_cap.isOpened():
        raise IOError(f"Could not open video: {video_path}")

    if not mask_cap.isOpened():
        raise IOError(f"Could not open mask video: {mask_path}")

    fps = video_cap.get(cv2.CAP_PROP_FPS)

    if fps <= 0:
        raise ValueError(f"Invalid FPS value for video: {fps}")

    frame_step = max(1, int(round(fps * sample_every_seconds)))

    frame_index = 0
    sample_index = 0
    saved_pairs = []

    video_stem = video_path.stem

    while True:
        video_success, video_frame = video_cap.read()
        mask_success, mask_frame = mask_cap.read()

        if not video_success or not mask_success:
            break

        if frame_index % frame_step == 0:
            image_name = f"{prefix}_{video_stem}_{sample_index:04d}{image_ext}"
            mask_name = f"{prefix}_{video_stem}_{sample_index:04d}{mask_ext}"

            image_output_path = output_images_dir / image_name
            mask_output_path = output_masks_dir / mask_name

            if len(mask_frame.shape) == 3:
                mask_frame = cv2.cvtColor(mask_frame, cv2.COLOR_BGR2GRAY)

            cv2.imwrite(str(image_output_path), video_frame)
            cv2.imwrite(str(mask_output_path), mask_frame)

            saved_pairs.append((image_output_path, mask_output_path))

            sample_index += 1

        frame_index += 1

    video_cap.release()
    mask_cap.release()

    return saved_pairs



## 1. pothole-dataset-1

After generating the initial segmentation masks, CVAT was used to manually review the mask quality. Each mask was assigned one of three review labels:

1. **OK**: The mask is accurate and can be used directly.
2. **Must Fix**: The mask contains annotation issues and needs correction before training.
3. **Must Delete**: The mask/image is unusable and should be removed from the dataset.

This review step helps improve dataset quality by keeping valid samples, identifying masks that require correction, and removing poor-quality annotations that could negatively affect model training.


In [ ]:
dataset_1_dir = DATA_INTERIM_DIR / 'pothole-dataset-1'
annotation_review_dir = REPORTS_DIR / 'data_review' / 'pothole-dataset-1' / 'annotation_review'
full_images_dir = dataset_1_dir / 'full_images'
full_masks_dir = dataset_1_dir / 'full_masks'

In [ ]:
need_fix_path = annotation_review_dir / 'images_need_fix.json'
need_fix_dir = dataset_1_dir / 'need_fix_images'

to_delete_path = annotation_review_dir / 'images_to_delete.json'

os.makedirs(need_fix_dir, exist_ok=True)

with open(need_fix_path, 'r', encoding='utf-8') as f:
    need_fix_json = json.load(f)

need_fix_count = need_fix_json['count']
need_fix_list = need_fix_json['images_need_fix']

print(f'Number of images that need fixing in pothole-dataset-1: {need_fix_count}')

for image_path in full_images_dir.iterdir():
    if image_path.name in need_fix_list:
        shutil.copy(image_path, need_fix_dir)

print(f'Successfully moved images from dataset pothole-dataset-1 that need fixing to {need_fix_dir}')

Number of images that need fixing in pothole-dataset-1: 183
Successfully moved images from dataset pothole-dataset-1 that need fixing to /home/omaralmadanii/projects/deep-learning/data_interim/pothole-dataset-1/need_fix_images


The masks marked as **Must Fix** during the CVAT review stage were manually corrected using CVAT. After correction, the updated masks were exported and stored in the `data_interim/pothole-dataset-1` directory under the `fixed_masks` folder.

These corrected masks replace the problematic initial annotations and are used to improve the quality and reliability of the final training dataset.


In [ ]:
filtered_images_dir = dataset_1_dir / 'filtered_images'
filtered_masks_dir = dataset_1_dir / 'filtered_masks'
fixed_masks_dir = dataset_1_dir / 'fixed_masks'

os.makedirs(filtered_images_dir, exist_ok=True)
os.makedirs(filtered_masks_dir, exist_ok=True)

with open(need_fix_path, 'r', encoding='utf-8') as f:
    need_fix_json = json.load(f)

with open(to_delete_path, 'r', encoding='utf-8') as f:
    to_delete_json = json.load(f)

need_fix_list = set(need_fix_json['images_need_fix'])
to_delete_list = set(to_delete_json['images_to_delete'])

overlap = need_fix_list & to_delete_list
if overlap:
    raise ValueError(
        f'pothole-dataset-1 has images marked as both need_fix and to_delete: {overlap}'
    )

for image_path in full_images_dir.iterdir():

    if not image_path.is_file():
        continue

    if image_path.name in to_delete_list:
        continue

    if image_path.name in need_fix_list:
        mask_path = fixed_masks_dir / f'{image_path.stem}.png'
    else:
        mask_path = full_masks_dir / f'{image_path.stem}.png'

    shutil.copy2(image_path, filtered_images_dir / image_path.name)
    shutil.copy2(mask_path, filtered_masks_dir / mask_path.name)

### Create CSV file for splits

In [ ]:
df1 = pd.DataFrame(columns=['Image', 'Mask', 'Split'])

images_list = os.listdir(dataset_1_dir / 'filtered_images')

df1['Image'] = images_list
df1['Mask'] = df1['Image'].str.replace('.jpg', '.png')

df1.head()

,Image,Mask,Split
0,pothole_dataset_1_train_image945.jpg,pothole_dataset_1_train_image945.png,NaN
1,pothole_dataset_1_valid_image1791.jpg,pothole_dataset_1_valid_image1791.png,NaN
2,pothole_dataset_1_valid_image1634.jpg,pothole_dataset_1_valid_image1634.png,NaN
3,pothole_dataset_1_train_image1283.jpg,pothole_dataset_1_train_image1283.png,NaN
4,pothole_dataset_1_train_image291.jpg,pothole_dataset_1_train_image291.png,NaN


In [ ]:
indices = range(len(df1))

train_indices, temp_indices = train_test_split(
    indices, test_size=0.2, random_state=42
)

val_indices, test_indices = train_test_split(
    temp_indices, test_size=0.5, random_state=42
)

In [ ]:
df1.loc[train_indices, 'Split'] = 'train'
df1.loc[val_indices, 'Split'] = 'val'
df1.loc[test_indices, 'Split'] = 'test'

In [ ]:
df1.head()

,Image,Mask,Split
0,pothole_dataset_1_train_image945.jpg,pothole_dataset_1_train_image945.png,train
1,pothole_dataset_1_valid_image1791.jpg,pothole_dataset_1_valid_image1791.png,train
2,pothole_dataset_1_valid_image1634.jpg,pothole_dataset_1_valid_image1634.png,val
3,pothole_dataset_1_train_image1283.jpg,pothole_dataset_1_train_image1283.png,train
4,pothole_dataset_1_train_image291.jpg,pothole_dataset_1_train_image291.png,train


In [ ]:
df1.to_csv(str(dataset_1_dir / 'splits.csv'), index=False)

## 2. pothole-dataset-2

In [ ]:
dataset_2_dir = DATA_INTERIM_DIR / 'pothole-dataset-2'
annotation_review_dir = REPORTS_DIR / 'data_review' / 'pothole-dataset-2' / 'annotation_review'
images_dir = dataset_2_dir / 'images'
masks_dir = dataset_2_dir / 'masks'

### 2.1 Generate Label Files

In [ ]:
labels_dir = dataset_2_dir / 'labels'
labels_dir.mkdir(exist_ok=True)

for mask_path in masks_dir.iterdir():
    mask_to_yolo(mask_path, labels_dir, mask_path.stem + '.txt')

After generating the initial segmentation masks, CVAT was used to manually review the mask quality. Each mask was assigned one of three review labels:

1. **OK**: The mask is accurate and can be used directly.
2. **Must Fix**: The mask contains annotation issues and needs correction before training.
3. **Must Delete**: The mask/image is unusable and should be removed from the dataset.

This review step helps improve dataset quality by keeping valid samples, identifying masks that require correction, and removing poor-quality annotations that could negatively affect model training.


In [ ]:
need_fix_path = annotation_review_dir / 'images_need_fix.json'
need_fix_dir = dataset_2_dir / 'need_fix_images'

to_delete_path = annotation_review_dir / 'images_to_delete.json'

os.makedirs(need_fix_dir, exist_ok=True)

with open(need_fix_path, 'r', encoding='utf-8') as f:
    need_fix_json = json.load(f)

need_fix_count = need_fix_json['count']
need_fix_list = need_fix_json['images_need_fix']

print(f'Number of images that need fixing in pothole-dataset-2: {need_fix_count}')

for image_path in images_dir.iterdir():
    if image_path.name in need_fix_list:
        shutil.copy(image_path, need_fix_dir)

print(f'Successfully moved images from dataset pothole-dataset-2 that need fixing to {need_fix_dir}')

Number of images that need fixing in pothole-dataset-2: 327
Successfully moved images from dataset pothole-dataset-2 that need fixing to /home/omaralmadanii/projects/deep-learning/data_interim/pothole-dataset-2/need_fix_images


In [ ]:
filtered_images_dir = dataset_2_dir / 'filtered_images'
filtered_masks_dir = dataset_2_dir / 'filtered_masks'
fixed_masks_dir = dataset_2_dir / 'fixed_masks'

os.makedirs(filtered_images_dir, exist_ok=True)
os.makedirs(filtered_masks_dir, exist_ok=True)

with open(need_fix_path, 'r', encoding='utf-8') as f:
    need_fix_json = json.load(f)

with open(to_delete_path, 'r', encoding='utf-8') as f:
    to_delete_json = json.load(f)

need_fix_list = set(need_fix_json['images_need_fix'])
to_delete_list = set(to_delete_json['images_to_delete'])

overlap = need_fix_list & to_delete_list
if overlap:
    raise ValueError(
        f'pothole-dataset-1 has images marked as both need_fix and to_delete: {overlap}'
    )

for image_path in images_dir.iterdir():

    if not image_path.is_file():
        continue

    if image_path.name in to_delete_list:
        continue

    if image_path.name in need_fix_list:
        mask_path = fixed_masks_dir / f'{image_path.stem}.png'
    else:
        mask_path = masks_dir / f'{image_path.stem}.png'

    shutil.copy2(image_path, filtered_images_dir / image_path.name)
    shutil.copy2(mask_path, filtered_masks_dir / mask_path.name)

### Create CSV file for splits

In [ ]:
df2 = pd.DataFrame(columns=['Image', 'Mask', 'Split'])

images_list = os.listdir(dataset_2_dir / 'filtered_images')

df2['Image'] = images_list
df2['Mask'] = df2['Image'].str.replace('.jpg', '.png')

df2.head()

,Image,Mask,Split
0,pothole_dataset_2_00009.jpg,pothole_dataset_2_00009.png,NaN
1,pothole_dataset_2_00943.jpg,pothole_dataset_2_00943.png,NaN
2,pothole_dataset_2_00237.jpg,pothole_dataset_2_00237.png,NaN
3,pothole_dataset_2_00937.jpg,pothole_dataset_2_00937.png,NaN
4,pothole_dataset_2_00365.jpg,pothole_dataset_2_00365.png,NaN


In [ ]:
indices = range(len(df2))

train_indices, temp_indices = train_test_split(
    indices, test_size=0.2, random_state=42
)

val_indices, test_indices = train_test_split(
    temp_indices, test_size=0.5, random_state=42
)

In [ ]:
df2.loc[train_indices, 'Split'] = 'train'
df2.loc[val_indices, 'Split'] = 'val'
df2.loc[test_indices, 'Split'] = 'test'

In [ ]:
df2.head()

,Image,Mask,Split
0,pothole_dataset_2_00009.jpg,pothole_dataset_2_00009.png,train
1,pothole_dataset_2_00943.jpg,pothole_dataset_2_00943.png,train
2,pothole_dataset_2_00237.jpg,pothole_dataset_2_00237.png,train
3,pothole_dataset_2_00937.jpg,pothole_dataset_2_00937.png,train
4,pothole_dataset_2_00365.jpg,pothole_dataset_2_00365.png,train


In [ ]:
df2.to_csv(str(dataset_2_dir / 'splits.csv'), index=False)

## 3. pothole-dataset-3

In [ ]:
dataset_3_dir = DATA_INTERIM_DIR / 'pothole-dataset-3'

In [ ]:
for split in ['train', 'val', 'test']:
    videos_dir = dataset_3_dir / split / 'videos'
    mask_videos_dir = dataset_3_dir / split / 'mask_videos'

    images_dir = dataset_3_dir / split / 'images'
    images_dir.mkdir(exist_ok=True)
    mask_images_dir = dataset_3_dir / split / 'mask_images'
    mask_images_dir.mkdir(exist_ok=True)

    for video_path in videos_dir.iterdir():
        mask_path = mask_videos_dir / video_path.name
        sample_video_and_mask_frames(
            video_path,
            mask_path,
            images_dir,
            mask_images_dir,
            sample_every_seconds=0.5,
            prefix='pothole_dataset_3_'
        )


### Create CSV file for splits

In [ ]:
df3 = pd.DataFrame(columns=['Image', 'Mask', 'Split'])

for split in ['train', 'val', 'test']:
    images_dir = dataset_3_dir / split / 'images'

    temp_df = pd.DataFrame(columns=['Image', 'Mask', 'Split'])
    temp_df['Image'] = os.listdir(images_dir)
    temp_df['Mask'] = temp_df['Image'].str.replace('.jpg', '.png')
    temp_df['Split'] = split

    df3 = pd.concat((df3, temp_df))

df3.head()

,Image,Mask,Split
0,pothole_dataset_3__0133_0000.png,pothole_dataset_3__0133_0000.png,train
1,pothole_dataset_3__0118_0000.png,pothole_dataset_3__0118_0000.png,train
2,pothole_dataset_3__0218_0003.png,pothole_dataset_3__0218_0003.png,train
3,pothole_dataset_3__0106_0003.png,pothole_dataset_3__0106_0003.png,train
4,pothole_dataset_3__0029_0002.png,pothole_dataset_3__0029_0002.png,train


In [ ]:
df3['Split'].value_counts()

Split
train    1488
val       496
test      492
Name: count, dtype: int64

In [ ]:
df3.to_csv(dataset_3_dir / 'splits.csv', index=False)

## Move to data_processed

In [ ]:
DATA_PROCESSED_DIR = PROJECT_DIR / 'data' / 'data_processed'
processed_images_dir = DATA_PROCESSED_DIR / 'images'
processed_masks_dir = DATA_PROCESSED_DIR / 'masks'

DATA_PROCESSED_DIR.mkdir(exist_ok=True)
processed_images_dir.mkdir(exist_ok=True)
processed_masks_dir.mkdir(exist_ok=True)

### Copy from pothole-dataset-1

In [ ]:
for image_path in (DATA_INTERIM_DIR / 'pothole-dataset-1' / 'filtered_images').iterdir():
    shutil.copy(image_path, processed_images_dir / image_path.name)

for mask_path in (DATA_INTERIM_DIR / 'pothole-dataset-1' / 'filtered_masks').iterdir():
    shutil.copy(mask_path, processed_masks_dir / mask_path.name)

### Copy from pothole-dataset-2

In [ ]:
for image_path in (DATA_INTERIM_DIR / 'pothole-dataset-2' / 'filtered_images').iterdir():
    shutil.copy(image_path, processed_images_dir / image_path.name)

for mask_path in (DATA_INTERIM_DIR / 'pothole-dataset-2' / 'filtered_masks').iterdir():
    shutil.copy(mask_path, processed_masks_dir / mask_path.name)

### Copy from pothole-dataset-3

In [ ]:
for split in ['train', 'val', 'test']:
    for image_path in (DATA_INTERIM_DIR / 'pothole-dataset-3' / split / 'images').iterdir():
        shutil.copy(image_path, processed_images_dir / image_path.name)

    for mask_path in (DATA_INTERIM_DIR / 'pothole-dataset-3' / split / 'mask_images').iterdir():
        shutil.copy(mask_path, processed_masks_dir / mask_path.name)

### Merge splits CSV files

In [ ]:
df1 = pd.read_csv(DATA_INTERIM_DIR / 'pothole-dataset-1' / 'splits.csv')
df2 = pd.read_csv(DATA_INTERIM_DIR / 'pothole-dataset-2' / 'splits.csv')
df3 = pd.read_csv(DATA_INTERIM_DIR / 'pothole-dataset-3' / 'splits.csv')

In [ ]:
final_df = pd.concat((df1, df2, df3))
final_df.to_csv(DATA_PROCESSED_DIR / 'splits.csv', index=False)

### Convert Masks to Binary

In [ ]:
for mask_path in (DATA_PROCESSED_DIR / 'masks').iterdir():
    binary_mask = convert_to_binary(mask_path)
    save_grayscale_image(binary_mask, DATA_PROCESSED_DIR / 'masks', mask_path.name)